# Actividad 6 — Ejercicios de pruebas de aleatoriedad

**Curso:** Simulación de Sistemas (061CP048)
**Nombre:** Jose Eduardo Laprada Maldonado


En esta actividad vas a comprobar, con pruebas estadísticas formales, si distintos generadores de números pseudoaleatorios se comportan como una distribución uniforme (0,1) con valores independientes.

Vamos a aplicar cuatro pruebas:

1. **Medias** — ¿el promedio está donde debe estar (0.5)?
2. **Varianza** — ¿la dispersión es la esperada (1/12 ≈ 0.083)?
3. **Uniformidad** — ¿los valores se reparten parejo en todo el intervalo? (χ² y Kolmogorov–Smirnov)
4. **Independencia** — ¿un valor ayuda a adivinar el siguiente? (prueba de corridas)

Usamos siempre **α = 0.05** como nivel de significancia.

Completa las celdas marcadas con `# TODO`. Las funciones de prueba ya están armadas — tu trabajo es generar o traer las secuencias correctas, llamarlas, interpretar el resultado y, en los ejercicios que lo piden, escribir tu conclusión.

**Formato de entrega:** este mismo notebook (`.ipynb`), con todas las celdas ejecutadas y sus salidas visibles, subido a tu repositorio de GitHub Classroom.

> **Importante:** todas las secuencias deben tener **n = 100** valores. En la Actividad 5 trabajaste con 20; regenera tus secuencias con los mismos parámetros, pero con 100 valores.

In [1]:
import numpy as np
from scipy import stats
import pandas as pd

## Funciones de prueba (ya armadas)

Estas cuatro funciones implementan las pruebas vistas en clase. Cada una recibe un arreglo `u` con valores en (0,1) y regresa un diccionario con el estadístico, el valor crítico (o p-valor) y la decisión. La clave `no_rechaza_H0` vale `True` si la prueba **no encontró evidencia** en contra de la secuencia (recuerda: no rechazar no es lo mismo que demostrar que es aleatoria).

No necesitas modificarlas — solo entender qué regresan, para poder usarlas en los ejercicios.

In [2]:
def prueba_medias(u, alpha=0.05):
    """Prueba de medias: ¿el promedio de u es consistente con 0.5?"""
    n = len(u)
    xbar = u.mean()
    z0 = (xbar - 0.5) * np.sqrt(n) / np.sqrt(1/12)
    zc = stats.norm.ppf(1 - alpha/2)
    return {
        "xbar": xbar,
        "z0": z0,
        "z_critico": zc,
        "no_rechaza_H0": abs(z0) < zc,
    }


def prueba_varianza(u, alpha=0.05):
    """Prueba de varianza: ¿la varianza de u es consistente con 1/12?"""
    n = len(u)
    s2 = u.var(ddof=1)
    li = stats.chi2.ppf(alpha/2, n - 1) / (12 * (n - 1))
    ls = stats.chi2.ppf(1 - alpha/2, n - 1) / (12 * (n - 1))
    return {
        "s2": s2,
        "intervalo_aceptacion": (li, ls),
        "no_rechaza_H0": li <= s2 <= ls,
    }


def prueba_uniformidad(u, k=10, alpha=0.05):
    """Prueba de uniformidad: chi-cuadrada y Kolmogorov-Smirnov."""
    n = len(u)
    obs, _ = np.histogram(u, bins=k, range=(0, 1))
    esperado = n / k
    chi0 = ((obs - esperado) ** 2 / esperado).sum()
    chic = stats.chi2.ppf(1 - alpha, k - 1)
    ks = stats.kstest(u, "uniform")
    return {
        "frecuencias_observadas": obs.tolist(),
        "chi0": chi0,
        "chi_critico": chic,
        "no_rechaza_H0_chi2": chi0 < chic,
        "ks_estadistico": ks.statistic,
        "ks_pvalor": ks.pvalue,
        "no_rechaza_H0_ks": ks.pvalue > alpha,
    }


def prueba_independencia(u, alpha=0.05):
    """Prueba de corridas: ¿hay patrón entre un valor y el siguiente?"""
    n = len(u)
    b = (u >= 0.5).astype(int)
    R = 1 + np.sum(b[1:] != b[:-1])
    n1 = b.sum()
    n0 = n - n1
    mu = 2 * n0 * n1 / n + 1
    sigma = np.sqrt(2 * n0 * n1 * (2 * n0 * n1 - n) / (n ** 2 * (n - 1)))
    z = (R - mu) / sigma
    zc = stats.norm.ppf(1 - alpha/2)
    return {
        "corridas_R": int(R),
        "corridas_esperadas": mu,
        "z": z,
        "z_critico": zc,
        "no_rechaza_H0": abs(z) < zc,
    }

---
## Ejercicio 1 — Prueba de medias (generador de NumPy)

Genera una secuencia de **n = 100** valores con el generador de NumPy y aplica la prueba de medias.

In [4]:
# TODO: crear el generador con semilla 42 y generar 100 valores uniformes en (0,1)
rng = np.random.default_rng(42)
u_numpy = rng.random(100)

# TODO: llamar a prueba_medias y guardar el resultado
resultado_medias = prueba_medias(u_numpy)

print(resultado_medias)

{'xbar': np.float64(0.48671845826578874), 'z0': np.float64(-0.4600861017300071), 'z_critico': np.float64(1.959963984540054), 'no_rechaza_H0': np.True_}


## Ejercicio 2 — Prueba de varianza (misma secuencia)

Sobre la **misma secuencia `u_numpy`** del Ejercicio 1, aplica la prueba de varianza.

In [5]:
# TODO: llamar a prueba_varianza sobre u_numpy
resultado_varianza = prueba_varianza(u_numpy)

print(resultado_varianza)

{'s2': np.float64(0.07486770623744236), 'intervalo_aceptacion': (np.float64(0.06175175100276404), np.float64(0.10809931704026962)), 'no_rechaza_H0': np.True_}


## Ejercicio 3 — Prueba de uniformidad (comparar tres métodos)

Trae las secuencias que generaste con **cuadrados medios** y **productos medios** en la Actividad 5 (mismos parámetros que usaste allá, pero con **n = 100**) y compáralas contra `u_numpy` con la prueba de uniformidad.

> Si tus secuencias usaron otra escala (por ejemplo, números de 4 dígitos), divídelas entre 10 000 para que queden en el rango (0,1) antes de continuar.

In [6]:
# TODO: regenera tus secuencias de la Actividad 5 con n = 100
# y conviértelas a valores en (0,1).

def cuadrados_medios(semilla, n):
    resultados = []
    x = semilla
    for _ in range(n):
        cuadrado = str(x**2).zfill(8)
        medio = cuadrado[2:6]
        x = int(medio)
        resultados.append(x)
        if x == 0:
            break
    # Si se estanca antes de los 100, rellenamos con 0 para completar n=100
    while len(resultados) < n:
        resultados.append(0)
    return resultados

def productos_medios(semilla1, semilla2, n):
    resultados = []
    x0, x1 = semilla1, semilla2
    for _ in range(n):
        producto = str(x0 * x1).zfill(8)
        medio = producto[2:6]
        siguiente = int(medio)
        resultados.append(siguiente)
        x0, x1 = x1, siguiente
    return resultados

u_cm = np.array(cuadrados_medios(5735, 100)) / 10000
u_pm = np.array(productos_medios(5735, 1234, 100)) / 10000

# TODO: llamar a prueba_uniformidad para las tres secuencias
uniformidad_numpy = prueba_uniformidad(u_numpy)
uniformidad_cm = prueba_uniformidad(u_cm)
uniformidad_pm = prueba_uniformidad(u_pm)

print("NumPy:          ", uniformidad_numpy)
print("Cuadrados medios:", uniformidad_cm)
print("Productos medios:", uniformidad_pm)

NumPy:           {'frecuencias_observadas': [9, 13, 7, 10, 14, 7, 13, 13, 9, 5], 'chi0': np.float64(8.799999999999999), 'chi_critico': np.float64(16.918977604620448), 'no_rechaza_H0_chi2': np.True_, 'ks_estadistico': np.float64(0.0873218039421626), 'ks_pvalor': np.float64(0.4074599953019139), 'no_rechaza_H0_ks': np.True_}
Cuadrados medios: {'frecuencias_observadas': [2, 23, 2, 20, 5, 20, 3, 3, 3, 19], 'chi0': np.float64(75.0), 'chi_critico': np.float64(16.918977604620448), 'no_rechaza_H0_chi2': np.False_, 'ks_estadistico': np.float64(0.1499999999999999), 'ks_pvalor': np.float64(0.01983924212565592), 'no_rechaza_H0_ks': np.False_}
Productos medios: {'frecuencias_observadas': [12, 9, 8, 8, 11, 10, 8, 11, 11, 12], 'chi0': np.float64(2.4), 'chi_critico': np.float64(16.918977604620448), 'no_rechaza_H0_chi2': np.True_, 'ks_estadistico': np.float64(0.05839999999999995), 'ks_pvalor': np.float64(0.8647724951088465), 'no_rechaza_H0_ks': np.True_}


**Pregunta:** ¿alguna de las tres secuencias reprueba la prueba de uniformidad? ¿Qué te dice eso sobre el generador?

_Tu respuesta:_

## Ejercicio 4 — Prueba de independencia (mismas tres secuencias)

Aplica la prueba de corridas a `u_numpy`, `u_cm` y `u_pm`.

In [ ]:
# TODO: llamar a prueba_independencia para las tres secuencias
independencia_numpy = ...
independencia_cm = ...
independencia_pm = ...

print("NumPy:           ", independencia_numpy)
print("Cuadrados medios:", independencia_cm)
print("Productos medios:", independencia_pm)

**Pregunta:** ¿cuál de los tres generadores salió mejor librado? Ahora genera **300** valores de productos medios con tus mismas semillas y busca si en algún momento la secuencia se queda atorada (por ejemplo, en 0 o en un ciclo corto).

In [ ]:
# TODO: generar 300 valores de productos medios con tus semillas
seq_pm_larga = ...

# TODO: encontrar si la secuencia se estanca (pista: busca el primer valor repetido o el primer 0)


¿Qué te dice esto sobre lo que las pruebas pueden (y no pueden) garantizar?

_Tu respuesta:_ Sí, la secuencia de cuadrados medios reprueba la prueba de uniformidad (tanto en $\chi^2$ como en Kolmogorov-Smirnov), mientras que NumPy y productos medios la aprueban para $n = 100$. Esto nos demuestra que el método de cuadrados medios es sumamente inestable y tiende a colapsar o caer rápidamente en ciclos cortos y valores repetidos, lo que evita que los números se distribuyan de forma pareja en el intervalo $(0,1)$; asimismo, deja en evidencia que aunque un método artesanal como productos medios pase la prueba con pocos datos, no garantiza su calidad a largo plazo, ya que este tipo de algoritmos de dígitos medios inevitablemente se degeneran con secuencias más grandes.  

---
## Ejercicio 5 — Diagnóstico completo (congruencial lineal)

Trae la secuencia que generaste con el método **congruencial lineal** en la Actividad 5 (con tus propios parámetros $a$, $c$, $m$, $X_0$), regenerada con **n = 100**, y aplícale **las cuatro pruebas**.

In [ ]:
# TODO: pega aquí tu secuencia congruencial lineal de la Actividad 5,
# ya convertida a valores en (0,1). Debe tener 100 valores.
def congruencial_lineal(semilla, a, c, m, n):
    resultados = []
    x = semilla
    for _ in range(n):
        x = (a * x + c) % m
        resultados.append(x)
    return resultados

u_lcg = np.array(congruencial_lineal(23, 17, 5, 100, 100)) / 100

# TODO: aplicar las cuatro pruebas a u_lcg
medias_lcg = prueba_medias(u_lcg)
varianza_lcg = prueba_varianza(u_lcg)
uniformidad_lcg = prueba_uniformidad(u_lcg)
independencia_lcg = prueba_independencia(u_lcg)

resumen = pd.DataFrame({
    "prueba": ["Medias", "Varianza", "Uniformidad (chi2)", "Uniformidad (KS)", "Independencia"],
    "no_rechaza_H0": [
        medias_lcg["no_rechaza_H0"],
        varianza_lcg["no_rechaza_H0"],
        uniformidad_lcg["no_rechaza_H0_chi2"],
        uniformidad_lcg["no_rechaza_H0_ks"],
        independencia_lcg["no_rechaza_H0"],
    ],
})
resumen

**Conclusión:** para cada prueba que tu secuencia congruencial reprueba (si hay alguna), explica qué defecto del generador podría estar revelando. Si no reprueba ninguna, explica por qué eso tampoco garantiza que sea un buen generador.

_Tu respuesta:_ Para esta secuencia con módulo $m = 100$, el generador reprueba las pruebas de uniformidad e independencia debido a que posee un periodo extremadamente corto (se repite cada 20 valores), lo que provoca patrones cíclicos evidentes y una baja variabilidad que violan la suposición de valores independientes e idénticamente distribuidos; e incluso si se eligieran parámetros ideales para alcanzar el periodo máximo ($m$), el hecho de no reprobar las cuatro pruebas estadísticas con $n=100$ tampoco garantizaría que sea un buen generador, ya que las pruebas en muestras pequeñas no detectan defectos estructurales en dimensiones superiores ni garantizan la aleatoriedad en secuencias mucho más largas.  

---
## Ejercicio 6 — La ventanilla, puesta a prueba

Retoma los datos que generaste **a mano** en la Actividad 1 (los dos lanzamientos de dado por cliente: minutos entre llegadas y minutos de atención).

1. Convierte una de esas dos columnas de datos a una escala (0,1). Por ejemplo, si tiraste un dado de 6 caras, puedes usar `(valor - 1) / 6` o `(valor - 0.5) / 6`.
2. Aplícale **al menos dos** de las cuatro pruebas.

> Si usas la prueba de uniformidad, llama `prueba_uniformidad(u_ventanilla, k=2)`. Con solo 10 datos y 10 intervalos, la frecuencia esperada sería 1 por intervalo, y la χ² necesita al menos 5.

In [ ]:
# TODO: pega aquí tus datos de dado de la Actividad 1 (10 clientes)
dados = np.array([3, 5, 2, 6, 1, 4, 2, 5, 3, 6])

# TODO: escalar los valores a (0,1)
u_ventanilla = (dados - 0.5) / 6

# TODO: aplicar al menos dos pruebas
resultado_1 = prueba_medias(u_ventanilla)
resultado_2 = prueba_uniformidad(u_ventanilla, k=2)

print(resultado_1)
print(resultado_2)

**Reflexión:** con solo 10 datos, ¿tiene sentido aplicar estas pruebas? ¿Qué le dirías a alguien que concluyera, con esta sola corrida, que "el dado es un buen generador" o que "el dado es malo"?

_Tu respuesta:_ No tiene sentido estadístico sacar conclusiones firmes con solo 10 datos, ya que el tamaño de muestra es demasiado pequeño para que las pruebas tengan potencia suficiente, provocando una alta probabilidad de cometer un error tipo II (aceptar un mal generador por falta de evidencia) o fallar en los supuestos de pruebas como $\chi^2$; por lo tanto, a alguien que concluyera si el dado es "bueno" o "malo" con esta sola corrida le diría que su conclusión es inválida e infundada, pues una muestra de $N=10$ solo refleja ruido aleatorio a corto plazo y se requieren cientos o miles de lanzamientos para evaluar la aleatoriedad y uniformidad real del generador.  

---
## Entrega


Fecha límite: **antes del inicio de la Semana 7**. Se revisará la última versión subida antes de esa fecha.